In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import json
import time
import joblib
import importlib.util

from pathlib import Path

from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT != PROJECT_ROOT.parent:

    expected_file = (
        PROJECT_ROOT /
        "Module_3_Feature_Engineering_Vectorization" /
        "features" /
        "ted_talks_features.csv"
    )

    if expected_file.exists():
        break

    PROJECT_ROOT = PROJECT_ROOT.parent

if not expected_file.exists():

    raise FileNotFoundError(
        "Could not locate Module_3_Feature_Engineering_Vectorization. "
        "Check your notebook location."
    )


MODULE_DIR = (
    PROJECT_ROOT /
    "Module_3_Feature_Engineering_Vectorization"
)

FEATURE_DIR = MODULE_DIR / "features"
MODEL_DIR = MODULE_DIR / "models"
SIMILARITY_DIR = MODULE_DIR / "similarity"

OUTPUT_DIR = MODULE_DIR / "outputs"

VIZ_DIR = OUTPUT_DIR / "visualizations"
REPORT_DIR = OUTPUT_DIR / "reports"
STATISTICS_DIR = OUTPUT_DIR / "statistics"

for folder in [
    MODEL_DIR,
    VIZ_DIR,
    REPORT_DIR,
    STATISTICS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


print("Project root:", PROJECT_ROOT)
print("Module directory:", MODULE_DIR)

Project root: C:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation
Module directory: C:\Users\DELL\Desktop\INTERNSHIPS\Infosys_SpringBoard\Ted_Talk_Recommendation\Module_3_Feature_Engineering_Vectorization


In [3]:
DATA_PATH = FEATURE_DIR / "ted_talks_features.csv"

TFIDF_PATH = FEATURE_DIR / "combined_tfidf_matrix.npz"

df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

tfidf_matrix = load_npz(
    TFIDF_PATH
).tocsr()

if len(df) != tfidf_matrix.shape[0]:

    raise ValueError(
        "Dataset and TF-IDF matrix are not aligned."
    )

print("Dataset shape:", df.shape)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\DELL\\Desktop\\INTERNSHIPS\\Infosys_SpringBoard\\Ted_Talk_Recommendation\\Module_3_Feature_Engineering_Vectorization\\features\\combined_tfidf_matrix.npz'

In [ ]:
text_columns = [
    "title_processed",
    "description_processed",
    "tags_processed",
    "transcript_processed"
]

if "combined_text" in df.columns:

    embedding_text = (
        df["combined_text"]
        .fillna("")
        .astype(str)
    )

else:

    available_columns = [
        column
        for column in text_columns
        if column in df.columns
    ]

    if not available_columns:

        raise ValueError(
            "No processed text columns available."
        )

    embedding_text = (
        df[available_columns]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )


# Replace empty text with a title-based fallback.
if "title_processed" in df.columns:

    empty_mask = (
        embedding_text.str.strip() == ""
    )

    embedding_text.loc[empty_mask] = (
        df.loc[empty_mask, "title_processed"]
        .fillna("")
        .astype(str)
    )

if (embedding_text.str.strip() == "").any():

    raise ValueError(
        "Some talks have no usable text for embedding."
    )

print("Text preparation completed.")

print("Number of documents:", len(embedding_text))

print("Example text:")

print(embedding_text.iloc[0][:500])

In [ ]:
import subprocess
import sys

if importlib.util.find_spec("sentence_transformers") is None:

    print("Installing Sentence Transformers...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "sentence-transformers"
    ])

else:

    print("Sentence Transformers is already installed.")

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    MODEL_NAME
)

print("Embedding model loaded successfully!")

In [ ]:
start_time = time.time()

embeddings = embedding_model.encode(
    embedding_text.tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embedding_time = time.time() - start_time

embeddings = embeddings.astype(np.float32)

print("Embedding generation completed!")

print("Embedding shape:", embeddings.shape)

print("Processing time:", round(embedding_time, 2), "seconds")

In [ ]:
EMBEDDING_PATH = FEATURE_DIR / "ted_talks_embeddings.npy"

MODEL_SAVE_PATH = MODEL_DIR / "sentence_transformer_model"

np.save(
    EMBEDDING_PATH,
    embeddings
)

embedding_model.save(
    str(MODEL_SAVE_PATH)
)

print("Embeddings saved:", EMBEDDING_PATH)

print("Model saved:", MODEL_SAVE_PATH)

In [ ]:
if embeddings.shape[0] != len(df):

    raise ValueError(
        "Embedding count does not match the TED Talk dataset."
    )

if embeddings.shape[1] != 384:

    raise ValueError(
        "Unexpected embedding dimension."
    )

if not np.isfinite(embeddings).all():

    raise ValueError(
        "Embeddings contain invalid numerical values."
    )

print("Embedding validation successful!")

print("Talks:", embeddings.shape[0])

print("Dimensions per talk:", embeddings.shape[1])

In [ ]:
def compare_similarity(
    talk_index,
    tfidf_matrix,
    embeddings,
    dataframe,
    top_n=10
):

    # TF-IDF similarity
    tfidf_scores = cosine_similarity(
        tfidf_matrix[talk_index],
        tfidf_matrix
    ).flatten()

    # Semantic embedding similarity
    semantic_scores = (
        embeddings @ embeddings[talk_index]
    )

    # Exclude the original talk
    tfidf_scores[talk_index] = -1

    semantic_scores[talk_index] = -1

    tfidf_indices = np.argsort(
        tfidf_scores
    )[::-1][:top_n]

    semantic_indices = np.argsort(
        semantic_scores
    )[::-1][:top_n]

    tfidf_results = pd.DataFrame({

        "method": "TF-IDF",

        "rank": range(1, top_n + 1),

        "recommended_index": tfidf_indices,

        "recommended_title": dataframe.iloc[
            tfidf_indices
        ]["title_processed"].values,

        "similarity_score": tfidf_scores[
            tfidf_indices
        ]

    })

    semantic_results = pd.DataFrame({

        "method": "Semantic Embeddings",

        "rank": range(1, top_n + 1),

        "recommended_index": semantic_indices,

        "recommended_title": dataframe.iloc[
            semantic_indices
        ]["title_processed"].values,

        "similarity_score": semantic_scores[
            semantic_indices
        ]

    })

    return pd.concat(
        [tfidf_results, semantic_results],
        ignore_index=True
    )

In [ ]:
sample_index = 0

comparison_results = compare_similarity(
    talk_index=sample_index,
    tfidf_matrix=tfidf_matrix,
    embeddings=embeddings,
    dataframe=df,
    top_n=10
)

print(
    "Selected TED Talk:",
    df.iloc[sample_index]["title_processed"]
)

display(comparison_results)

In [ ]:
tfidf_recommendations = set(
    comparison_results[
        comparison_results["method"] == "TF-IDF"
    ]["recommended_index"]
)

semantic_recommendations = set(
    comparison_results[
        comparison_results["method"] == "Semantic Embeddings"
    ]["recommended_index"]
)

common_recommendations = (
    tfidf_recommendations &
    semantic_recommendations
)

overlap_count = len(common_recommendations)

overlap_percentage = (
    overlap_count / 10
) * 100

print("Common recommendations:", overlap_count)

print("Top-10 overlap:", overlap_percentage, "%")

print("Common recommended talks:")

display(
    df.iloc[
        sorted(common_recommendations)
    ][["title_processed"]]
)

In [ ]:
plot_data = comparison_results.copy()

plt.figure(figsize=(12, 7))

sns.barplot(
    data=plot_data,
    x="similarity_score",
    y="recommended_title",
    hue="method"
)

plt.title(
    "TF-IDF vs Semantic Embedding Recommendations"
)

plt.xlabel("Cosine Similarity")

plt.ylabel("Recommended TED Talks")

plt.tight_layout()

plt.savefig(
    VIZ_DIR / "tfidf_vs_semantic_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
tfidf_size_mb = (
    TFIDF_PATH.stat().st_size / (1024 ** 2)
)

embedding_size_mb = (
    EMBEDDING_PATH.stat().st_size / (1024 ** 2)
)

comparison_statistics = pd.DataFrame({

    "Metric": [
        "Number of TED Talks",
        "TF-IDF Features",
        "Embedding Dimensions",
        "TF-IDF Storage (MB)",
        "Embedding Storage (MB)",
        "Embedding Generation Time (seconds)"
    ],

    "Value": [

        len(df),

        tfidf_matrix.shape[1],

        embeddings.shape[1],

        round(tfidf_size_mb, 2),

        round(embedding_size_mb, 2),

        round(embedding_time, 2)

    ]

})

display(comparison_statistics)

In [ ]:
comparison_report = {

    "dataset": {

        "total_talks": int(len(df))

    },

    "tfidf": {

        "feature_dimensions": int(
            tfidf_matrix.shape[1]
        ),

        "storage_mb": round(
            tfidf_size_mb,
            2
        )

    },

    "semantic_embeddings": {

        "model": MODEL_NAME,

        "embedding_dimensions": int(
            embeddings.shape[1]
        ),

        "storage_mb": round(
            embedding_size_mb,
            2
        ),

        "generation_time_seconds": round(
            embedding_time,
            2
        )

    },

    "recommendation_comparison": {

        "top_n": 10,

        "common_recommendations": int(
            overlap_count
        ),

        "overlap_percentage": float(
            overlap_percentage
        )

    }

}

with open(
    REPORT_DIR / "vectorization_comparison_report.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        comparison_report,
        file,
        indent=4
    )

comparison_statistics.to_csv(
    STATISTICS_DIR / "vectorization_comparison.csv",
    index=False
)

comparison_results.to_csv(
    SIMILARITY_DIR / "tfidf_vs_semantic_recommendations.csv",
    index=False
)

print("Comparison reports saved successfully!")